In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from src.model_loader import load_model, get_device
from src.hooks import extract_activations

In [2]:
model, tokenizer, device = load_model(
    "open-unlearning/tofu_Llama-3.2-1B-Instruct_full"
)

df = pd.read_csv("../results/cleaned_harmful_harmless_responses_labeled.csv")

refused_harmful = df[(df["type"] == "harmful") & (df["refused"] == True)]
harmless = df[df["type"] == "harmless"]

refused_prompts = refused_harmful["prompt"].tolist()
harmless_prompts = harmless["prompt"].tolist()

print(f"Refused harmful prompts: {len(refused_prompts)}")
print(f"Harmless prompts: {len(harmless_prompts)}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded: open-unlearning/tofu_Llama-3.2-1B-Instruct_full
Device: mps | dtype: torch.float16
Params: 1.2B
Refused harmful prompts: 35
Harmless prompts: 50


In [3]:
LAYER = "model.layers.14"

print("Extracting refused harmful activations...")
refused_acts = extract_activations(model, tokenizer, refused_prompts, LAYER, device)

print("Extracting harmless activations...")
harmless_acts = extract_activations(model, tokenizer, harmless_prompts, LAYER, device)

print(f"\nRefused acts shape: {refused_acts.shape}")
print(f"Harmless acts shape: {harmless_acts.shape}")

Extracting refused harmful activations...
Extracting harmless activations...

Refused acts shape: (35, 2048)
Harmless acts shape: (50, 2048)


In [4]:
# Difference in means — Arditi et al. methodology
refusal_direction = refused_acts.mean(axis=0) - harmless_acts.mean(axis=0)
refusal_direction = refusal_direction / np.linalg.norm(refusal_direction)

print(f"Refusal direction shape: {refusal_direction.shape}")
print(f"Refusal direction norm: {np.linalg.norm(refusal_direction):.6f}")

# Save for use in ablation notebook
np.save("../data/refusal_direction_layer14.npy", refusal_direction)
print("Saved refusal direction.")

Refusal direction shape: (2048,)
Refusal direction norm: 1.000000
Saved refusal direction.


In [5]:
refused_proj = refused_acts @ refusal_direction
harmless_proj = harmless_acts @ refusal_direction

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=refused_proj, name="Refused harmful",
    opacity=0.65, marker_color="#ef4444", nbinsx=30,
))
fig.add_trace(go.Histogram(
    x=harmless_proj, name="Harmless",
    opacity=0.65, marker_color="#3b82f6", nbinsx=30,
))
fig.update_layout(
    barmode="overlay",
    title="Distribution of activations along refusal direction (Layer 14)",
    xaxis_title="Projection value",
    yaxis_title="Count",
    template="plotly_white",
    width=800, height=400,
)
fig.show()